<a href="https://colab.research.google.com/github/limammahmoudabdelkader-web/AGRO-DATA/blob/main/LIMAM%20DRUG%20DESIGN%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

FROM GIT HUB

Ce notebook a pour objectif de configurer la bibliothèque DeepChem pour concevoir et entraîner un modèle d'apprentissage profond (DAGModel) afin de prédire la solubilité des molécules à partir du jeu de données Delaney (ESOL).

Voici une explication détaillée de chaque section de votre notebook, ainsi que des erreurs rencontrées :

1. Installation de l'environnement (Cellules bFoC5Zsa0gph à 0nkXFWmg5VfO)
Conda et DeepChem : La cellule bFoC5Zsa0gph tente de télécharger et d'installer Miniconda, puis d'installer DeepChem. Cependant, Conda a échoué car les conditions d'utilisation d'Anaconda n'ont pas été acceptées (CondaToSNonInteractiveError).
GPU : La cellule g5SoI1sm06x5 vérifie avec succès que TensorFlow détecte bien votre carte graphique (GPU).
Git Clone & Compilation : La cellule yn4NG37B2HGT clone le dépôt de DeepChem. Le répertoire /content/deepchem existant déjà, les commandes suivantes ont changé de dossier vers deepchem/deepchem/ au lieu de la racine du dépôt, ce qui a causé l'échec de l'installation (python setup.py install introuvable).
2. Chargement des données (Cellules TAqyRKzg5cTV et aYHN_QAX5d_Q)
Le problème AttributeError : L'import de Deepchem a réussi, mais l'appel à dc.molnet a échoué car DeepChem n'a pas été installé correctement à l'étape précédente. Normalement, cette étape charge le jeu de données Delaney, le sépare en ensembles d'entraînement, de validation et de test, et applique un featuriseur adapté.
3. Préparation des données & Transformation (Cellule yL5Hxzvm7CPs)
Cette cellule prépare les données pour l'apprentissage. Elle calcule le nombre maximum d'atomes par molécule et utilise un DAGTransformer pour convertir les structures moléculaires en graphes acycliques dirigés (Directed Acyclic Graphs), qui est le format requis par le modèle DAG.
4. Entraînement et Évaluation progressive (Cellules sHJ3KzSy7RQ5 à lnEe180E0cSe)
Initialisation du modèle : Vous définissez un modèle de graphes orientés DAGModel pour de la régression.
Entraînement incrémental : Le notebook contient des phases successives d'entraînement (model.fit) avec un nombre d'époques croissant (10, 100, 500, 2000, puis 5000 époques) entrecoupées par des étapes d'évaluation pour observer la performance du modèle sur les ensembles d'entraînement et de validation grâce à la métrique du coefficient de Pearson ($R^2$$R^2$).

In [7]:
!wget -c https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
!chmod +x Miniconda3-latest-Linux-x86_64.sh
!bash ./Miniconda3-latest-Linux-x86_64.sh -b -f -p /usr/local
!conda install -y -c deepchem -c rdkit -c conda-forge -c omnia deepchem-gpu=2.3.0
import sys
sys.path.append('/usr/local/lib/python3.7/site-packages/')

--2026-10-18 21:07:48--  https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
Resolving repo.anaconda.com (repo.anaconda.com)... 104.16.191.158, 104.16.32.241, 2606:4700::6810:bf9e, ...
Connecting to repo.anaconda.com (repo.anaconda.com)|104.16.191.158|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 216646594 (207M) [application/octet-stream]
Saving to: ‘Miniconda3-latest-Linux-x86_64.sh’

Miniconda3-latest-L 100%[===================>] 206.61M   221MB/s    in 0.9s    

2026-10-18 21:07:49 (221 MB/s) - ‘Miniconda3-latest-Linux-x86_64.sh’ saved [216646594/216646594]

PREFIX=/usr/local
Unpacking bootstrapper...
Unpacking payload...

Installing base environment...

Preparing transaction: ...working... done
Executing transaction: ...working... done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    For best results, please

In [8]:
import tensorflow as tf
print("tf.__version__: %s" % str(tf.__version__))
device_name = tf.test.gpu_device_name()
if not device_name:
  raise SystemError('GPU device not found')
print('Found GPU at: {}'.format(device_name))

tf.__version__: 2.21.0
Found GPU at: /device:GPU:0


In [9]:
!git clone https://github.com/deepchem/deepchem.git

fatal: destination path 'deepchem' already exists and is not an empty directory.


In [10]:
%cd deepchem/
!ls

/content/deepchem/deepchem
data  feat   __init__.py   metrics  molnet	 rl	 trans
dock  hyper  metalearning  models   __pycache__  splits  utils


In [11]:
!python setup.py install

python: can't open file '/content/deepchem/deepchem/setup.py': [Errno 2] No such file or directory


In [12]:
import deepchem as dc

In [13]:
delaney_tasks, delaney_datasets, transformers = dc.molnet.load_delaney(
    featurizer='GraphConv', split='index')
train_dataset, valid_dataset, test_dataset = delaney_datasets

AttributeError: module 'deepchem' has no attribute 'molnet'

In [ ]:
# Set up to Fit models
metric = dc.metrics.Metric(dc.metrics.pearson_r2_score, np.mean)

n_atom_feat = 75
batch_size = 64

max_atoms_train = max([mol.get_num_atoms() for mol in train_dataset.X])
max_atoms_valid = max([mol.get_num_atoms() for mol in valid_dataset.X])
max_atoms_test = max([mol.get_num_atoms() for mol in test_dataset.X])
max_atoms = max([max_atoms_train, max_atoms_valid, max_atoms_test])

reshard_size = 512
transformer = dc.trans.DAGTransformer(max_atoms=max_atoms)
train_dataset.reshard(reshard_size)
train_dataset = transformer.transform(train_dataset)
valid_dataset.reshard(reshard_size)
valid_dataset = transformer.transform(valid_dataset)

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)

model = dc.models.DAGModel(
    len(delaney_tasks),
    max_atoms=max_atoms,
    n_atom_feat=n_atom_feat,
    batch_size=batch_size,
    learning_rate=1e-3,
    use_queue=False,
    mode='regression')

# Fit trained model
model.fit(train_dataset, nb_epoch=10)

In [ ]:
print("Evaluating model")
train_scores = model.evaluate(train_dataset, [metric], transformers)
valid_scores = model.evaluate(valid_dataset, [metric], transformers)

print("Train scores")
print(train_scores)

print("Validation scores")
print(valid_scores)

In [ ]:
# Fit trained model
model.fit(train_dataset, nb_epoch=100)

In [ ]:
print("Evaluating model")
train_scores = model.evaluate(train_dataset, [metric], transformers)
valid_scores = model.evaluate(valid_dataset, [metric], transformers)

print("Train scores")
print(train_scores)

print("Validation scores")
print(valid_scores)

In [ ]:
# Fit trained model
model.fit(train_dataset, nb_epoch=500)

In [ ]:
print("Evaluating model")
train_scores = model.evaluate(train_dataset, [metric], transformers)
valid_scores = model.evaluate(valid_dataset, [metric], transformers)

print("Train scores")
print(train_scores)

print("Validation scores")
print(valid_scores)

In [ ]:
# Fit trained model
model.fit(train_dataset, nb_epoch=500)

In [ ]:
print("Evaluating model")
train_scores = model.evaluate(train_dataset, [metric], transformers)
valid_scores = model.evaluate(valid_dataset, [metric], transformers)

print("Train scores")
print(train_scores)

print("Validation scores")
print(valid_scores)

In [ ]:
# Fit trained model
model.fit(train_dataset, nb_epoch=2000)

In [ ]:
print("Evaluating model")
train_scores = model.evaluate(train_dataset, [metric], transformers)
valid_scores = model.evaluate(valid_dataset, [metric], transformers)

print("Train scores")
print(train_scores)

print("Validation scores")
print(valid_scores)

In [ ]:
# Fit trained model
model.fit(train_dataset, nb_epoch=5000)